*Synthetic data for method development—not real operational or clinical data.*


# Line B Yield Ingest — MFG-LINEB-2026

Manufacturing data science notebook for **Line B first-pass yield** ingestion and batch cleaning.
Project `MFG-LINEB-2026` covers Q1 2026 semiconductor assembly batches exported from the MES historian.

This notebook loads raw yield extracts, validates schema, and produces a cleaned batch table for downstream SPC.
Continue SPC analysis in `synthetic_mfg_yield_spc.ipynb`.


In [2]:
import numpy as np
import pandas as pd

project_id = 'MFG-LINEB-2026'
raw_csv_path = 'line_b_yield_MFG-LINEB-2026.csv'
expected_columns = ['batch_id', 'tool_id', 'yield_pct', 'timestamp']

print(f'project_id={project_id}')
print(f'raw_csv_path={raw_csv_path}')
print(f'expected_columns={",".join(expected_columns)}')


project_id=MFG-LINEB-2026
raw_csv_path=line_b_yield_MFG-LINEB-2026.csv
expected_columns=batch_id,tool_id,yield_pct,timestamp


## Raw Extract Load

The MES export for Line B is a flat CSV keyed by `batch_id`. Each row records the finishing tool assignment
and measured first-pass yield percentage at lot close.


In [3]:
raw_df = pd.read_csv(raw_csv_path, parse_dates=['timestamp'])
missing_cols = [col for col in expected_columns if col not in raw_df.columns]
if missing_cols:
    raise ValueError(f'missing columns: {missing_cols}')

print(f'raw_rows={len(raw_df)}')
print(f'tool_ids={sorted(raw_df.tool_id.unique())}')
print(raw_df.head(4).to_string(index=False))


raw_rows=30
tool_ids=['Tool_1', 'Tool_2', 'Tool_3', 'Tool_4']
  batch_id tool_id  yield_pct           timestamp
LINEB-B001  Tool_1      93.67 2026-01-06 06:00:00
LINEB-B002  Tool_1      94.46 2026-01-06 10:00:00
LINEB-B003  Tool_1      94.84 2026-01-06 14:00:00
LINEB-B004  Tool_1      95.64 2026-01-06 18:00:00


## Batch Cleaning Rules

Duplicate `batch_id` rows are dropped, yields outside `[70, 100]` are flagged, and timestamps are normalized to UTC-naive
for alignment with the plant historian. The helper `clean_line_b_batches` encapsulates these rules.


In [4]:
def clean_line_b_batches(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize Line B yield batches for SPC handoff."""
    cleaned = frame.drop_duplicates(subset=['batch_id']).copy()
    cleaned['yield_pct'] = cleaned['yield_pct'].astype(float)
    cleaned = cleaned[(cleaned['yield_pct'] >= 70.0) & (cleaned['yield_pct'] <= 100.0)]
    cleaned['timestamp'] = pd.to_datetime(cleaned['timestamp'])
    cleaned = cleaned.sort_values('timestamp').reset_index(drop=True)
    return cleaned

cleaned_df = clean_line_b_batches(raw_df)
print(f'cleaned_rows={len(cleaned_df)}')
print(f'yield_pct_mean={cleaned_df.yield_pct.mean():.2f}')
print(cleaned_df.groupby('tool_id')['yield_pct'].mean().round(2).to_string())


cleaned_rows=30
yield_pct_mean=93.02
tool_id
Tool_1    94.35
Tool_2    93.51
Tool_3    89.41
Tool_4    93.91


## Ingest Manifest

The historian manifest tracks all Line B lots received this quarter, including rows held for manual QC review.
Only cleaned rows proceed to SPC; manifest row count includes pending lots not yet in the CSV extract.


In [5]:
ingest_manifest_rows = 4800
pending_qc_rows = 112
exported_rows = len(cleaned_df)

print(f'ingest_manifest_rows={ingest_manifest_rows}')
print(f'pending_qc_rows={pending_qc_rows}')
print(f'exported_rows={exported_rows}')
print(f'project_id={project_id}')


ingest_manifest_rows=4800
pending_qc_rows=112
exported_rows=30
project_id=MFG-LINEB-2026


## Handoff Notes

Cleaned batches are written to the shared analytics bucket as `line_b_cleaned.parquet` (not committed here).
**Next step:** open `synthetic_mfg_yield_spc.ipynb` for Western Electric rule evaluation on the cleaned yield series.

Do not infer root-cause conclusions from ingest alone — tool-level drift requires SPC and engineering review.
